# TabFM vs Meridian — gap analysis (not a bake-off)

**Practitioner takeaway: predictive fit ≠ media decisioning.**

> **Disclaimer:** Official Meridian **simulated/demo** data only. Not real campaign
> performance. Tiny MCMC is **directional / not decision-grade**.

This notebook:
1. States the deliverables gap honestly (only predictive KPI is a Yes for TabFM)
2. Loads one frozen official Meridian-schema table
3. Scores **both** models on the **same** later-week holdout (Meridian `holdout_id`)
4. Shows Meridian-only contribution / ROI tables separately (not peer-scored vs TabFM ablation)

TabFM weight license: `tabfm-non-commercial-v1.0`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "mmm_compare").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from mmm_compare.compare import print_table, run_comparison
from mmm_compare.data import DEFAULT_RUNTIME_DATASET, PREFERRED_OFFICIAL, load_mmm_dataset
from mmm_compare.deliverables import capability_summary, deliverables_dataframe

RESULTS = ROOT / "results"
# Preferred official = geo; runtime default on CPU = national
DATASET = DEFAULT_RUNTIME_DATASET
DRY_RUN = True
ABLATION_PROXY = False  # keep False unless inspecting quarantined sensitivity hack

print("Preferred official:", PREFERRED_OFFICIAL)
print("Runtime dataset:", DATASET)
print("DRY_RUN:", DRY_RUN, "| ABLATION_PROXY:", ABLATION_PROXY)

## 1) Deliverables matrix — Meridian product surface vs TabFM

In [ ]:
caps = capability_summary()
print(caps["headline"])
print("Counts:", caps["counts"])
display(deliverables_dataframe())

## 2) Frozen official input + shared holdout

In [ ]:
data = load_mmm_dataset(dataset=DATASET, max_context_rows=100)
print("Source:", data.source_path)
print("KPI:", data.kpi_col)
print("Features (media/controls):", data.feature_cols)
print("Train / holdout rows:", len(data.train_idx), len(data.test_idx))
print("TabFM context rows:", len(data.tabfm_context_idx))
print("holdout_id shape:", data.holdout_id.shape)
print("Holdout times (head):", data.holdout_times[:5], "...")
print("Notes:", data.notes)
display(data.frame.head())

## 3) Fair OOS predictive KPI (headline comparison)

In [ ]:
table, results, payload = run_comparison(
    dataset=DATASET,
    dry_run=DRY_RUN,
    prefer_real_meridian=True,
    results_dir=RESULTS,
    ablation_proxy=ABLATION_PROXY,
)
print_table(table)
display(table)
print("Takeaway:", payload["practitioner_takeaway"])
print("Fair eval:", payload["fair_eval"]["scoring"])
print("Remaining asymmetry:", payload["fair_eval"]["remaining_asymmetry"])
print("Modes:", results["tabfm"].mode, results["meridian"].mode)

## 4) Meridian-only deliverables (not TabFM peers)

Contribution / ROI / response curves are **Meridian product surface**. They are shown
for inspection, not as a side-by-side against TabFM ablation.

In [ ]:
mer = payload["deliverable_tables"]["meridian_only_product_surface"]
print("MCMC quality:", mer.get("mcmc_quality"))

print("=== predictive_accuracy (Train/Test when holdout_id set) ===")
display(pd.DataFrame(mer.get("predictive_accuracy") or []))

print("=== Meridian incremental contribution ===")
display(pd.DataFrame(
    [{"channel": k, "incremental_outcome": v}
     for k, v in (mer.get("channel_contribution") or {}).items()]
))

print("=== Meridian ROI ===")
display(pd.DataFrame(
    [{"channel": k, "roi": v} for k, v in (mer.get("roi_by_channel") or {}).items()]
))

print("=== TabFM honesty ===")
print(payload["deliverable_tables"]["tabfm"].get("honesty"))
abl = payload["deliverable_tables"]["tabfm"].get("ablation_proxy_NOT_meridian_equivalent")
if abl:
    print("Ablation proxy present (quarantined):", abl.get("warning"))
else:
    print("Ablation proxy not enabled (default).")

In [ ]:
import matplotlib.pyplot as plt

n = min(len(results["tabfm"].y_pred_test), len(results["meridian"].y_pred_test), 60)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(n), results["tabfm"].y_pred_test[:n], marker="s", label=f"TabFM ({results['tabfm'].mode})")
ax.plot(range(n), results["meridian"].y_pred_test[:n], marker="^", label=f"Meridian ({results['meridian'].mode})")
ax.set_title("Shared-holdout KPI predictions (simulated data; predictive fit only)")
ax.set_ylabel(data.kpi_col)
ax.legend()
fig.tight_layout()
plt.show()

## Run with real backends

```bash
python scripts/run_comparison.py -v
python scripts/run_comparison.py --dataset geo -v   # preferred official file; geo MCMC may skip
```

Set `DRY_RUN = False` after TabFM weights + Meridian are installed.